# CNZS Pilot Test Survey Quantitative Analysis

## Step 0: Imports & plotting settings

In [1]:
# ============================================================
# STEP 0: IMPORTS — Core Libraries for Quantitative Analysis & Plotting
# ============================================================
import os
import re
import ast
import json
import textwrap
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from cycler import cycler
from matplotlib.colors import ListedColormap
from typing import List
from matplotlib.cm import get_cmap

In [2]:
# ============================================================
# PLOTTING SETTINGS — Consistent Style Across All Charts
# ============================================================

# Custom CNZS / SBTi color palette (your exact colors)
custom_colors = [
    '#5D266D',  # Deep Purple
    '#12284C',  # Navy Blue
    '#D77932',  # Orange
    '#CF4143',  # Red
    '#3289B7',  # Sky Blue
    '#344EA1',  # Indigo Blue
    '#000000',  # Black
    "#5D5D5D66" # Soft Grey
]

# Apply global color cycle
plt.rcParams['axes.prop_cycle'] = cycler(color=custom_colors)

# Typography settings
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'

# Remove unnecessary chart borders
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.left'] = False

# Grid styling
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.color'] = 'grey'

# Text + axis colors
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['axes.edgecolor'] = 'grey'

# Show applied palette
print("Active color cycle:", plt.rcParams['axes.prop_cycle'].by_key()['color'])


Active color cycle: ['#5D266D', '#12284C', '#D77932', '#CF4143', '#3289B7', '#344EA1', '#000000', '#5D5D5D66']


In [3]:
# ============================================================
# LOAD CLEANED *QUANTITATIVE ONLY* DATA 
# ============================================================

file_path = (
    "../output/PC2_all_quantitative_responses_one_answer_per_line_for_analysis.xlsx"
)

df = pd.read_excel(file_path, dtype={"Reference Criteria": str})

print(f"📄 Loaded quantitative dataset: {df.shape}")
print("Columns:", list(df.columns), "...")

📄 Loaded quantitative dataset: (90676, 24)
Columns: ['Response ID', 'Country', 'Sector', 'SBTi Status', 'Employees MRY', 'Annual Turnover MRY', 'Category A/B', 'Stakeholder Category', 'World Bank Income Group', 'Region', 'Question', 'Answer', 'Question ID', 'OG Question Number', 'Section', 'Subsection', 'Mapping Group', 'Question Group', 'Reference Criteria', 'Taskforce Person', 'Content Reference', 'Q Type', 'Group Quanti?', 'Question Options'] ...


In [4]:
df.head()

,Response ID,Country,Sector,SBTi Status,Employees MRY,Annual Turnover MRY,Category A/B,Stakeholder Category,World Bank Income Group,Region,...,Section,Subsection,Mapping Group,Question Group,Reference Criteria,Taskforce Person,Content Reference,Q Type,Group Quanti?,Question Options
0,115004127093,United States,NaN,N/A - Not relevant to me,N/A - Not relevant to me,N/A - Not relevant to me,Category A,Academia,High income,Northern America,...,Introduction,General questions,"235,236","17,18",Not applicable,Jasraj,"Emma Watson, Alice Farrelly",quanti,n,"[""Strongly agree"", ""Somewhat agree"", ""Neutral""..."
1,115004217026,Netherlands,NaN,N/A - Not relevant to me,N/A - Not relevant to me,N/A - Not relevant to me,Category A,Industry associations,High income,Europe,...,Introduction,General questions,"235,236","17,18",Not applicable,Jasraj,"Emma Watson, Alice Farrelly",quanti,n,"[""Strongly agree"", ""Somewhat agree"", ""Neutral""..."
2,114997732177,United States,NaN,N/A - Not relevant to me,N/A - Not relevant to me,N/A - Not relevant to me,Category A,Civil society,High income,Northern America,...,Introduction,General questions,"235,236","17,18",Not applicable,Jasraj,"Emma Watson, Alice Farrelly",quanti,n,"[""Strongly agree"", ""Somewhat agree"", ""Neutral""..."
3,115003959923,Singapore,NaN,N/A - Not relevant to me,N/A - Not relevant to me,N/A - Not relevant to me,Category A,Other,High income,Asia,...,Introduction,General questions,"235,236","17,18",Not applicable,Jasraj,"Emma Watson, Alice Farrelly",quanti,n,"[""Strongly agree"", ""Somewhat agree"", ""Neutral""..."
4,115002532209,Mexico,Electrical Equipment and Machinery,My company has a validated net-zero science-ba...,"More than 1,000",N/A - Not relevant to me,Category A,Industry,Upper middle income,Latin America and the Caribbean,...,Introduction,General questions,"235,236","17,18",Not applicable,Jasraj,"Emma Watson, Alice Farrelly",quanti,n,"[""Strongly agree"", ""Somewhat agree"", ""Neutral""..."


## Step 1: Import and preprocess

In [5]:
# ============================================================
# Step 1: PRE-PROCESS ANSWERS & STANDARDIZE METADATA
# ============================================================

# ------------------------------------------------------------
# 1. Normalize free-text answers (Answer column)
# ------------------------------------------------------------
def normalize_text(val):
    """
    Clean up free-text values:
      - Strip leading/trailing whitespace
      - Collapse multiple spaces
      - Lowercase then capitalize first letter
    """
    if isinstance(val, str):
        return " ".join(val.strip().lower().split()).capitalize()
    return val

# Inspect columns if needed
# display(df.columns)

# Apply normalization to all answers
df["Answer"] = df["Answer"].apply(normalize_text)


# ------------------------------------------------------------
# 2. Convert rank-like numeric answers to integer strings
# ------------------------------------------------------------
def convert_to_int_str(x):
    """
    If x looks like a number (e.g. '1', '1.0', 2.0),
    convert to an integer and then to string.
    Otherwise return as-is.
    """
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return x

df["Answer"] = df["Answer"].apply(convert_to_int_str)


# ------------------------------------------------------------
# 3. Standardize SBTi Status values
# ------------------------------------------------------------
df["SBTi Status"] = df["SBTi Status"].astype(str).str.strip().str.lower()

sbti_mapping_normalized = {
    "my company has a validated net-zero science-based target": "NZ only",
    "my company has a validated near-term science-based target": "NT only",
    "my company has a commitment to set a science-based target": "Commitment",
    "my company has not committed to set science-based targets and doesn't have a validated target": "None",
}

df["SBTi Status"] = df["SBTi Status"].replace(sbti_mapping_normalized)


# ------------------------------------------------------------
# 4. Standardize Region labels
# ------------------------------------------------------------
df["Region"] = df["Region"].replace({
    "Latin America and the Caribbean": "LATAM and Caribbean",
    "Northern America": "North America",
    "MENA": "Middle East",
})


# ------------------------------------------------------------
# 5. Ensure Question ID is numeric
# ------------------------------------------------------------
df["Question ID"] = pd.to_numeric(df["Question ID"], errors="coerce")


# ------------------------------------------------------------
# 6. Clean Category A/B field + SBTi NAs
# ------------------------------------------------------------
# Rename Category A/B → Category AB for easier handling
df.rename(columns={"Category A/B": "Category AB"}, inplace=True)

# Fill missing categories with explicit N/A
df["Category AB"] = df["Category AB"].replace(np.nan, "N/A")

# Some SBTi entries may be literal 'nan' strings → turn into N/A
df["SBTi Status"] = df["SBTi Status"].replace("nan", "N/A")


# ------------------------------------------------------------
# 7. Metadata columns + plotting helpers
# ------------------------------------------------------------
metadata_columns = ["Region", "SBTi Status", "Category AB", "Stakeholder Category", "World Bank Income Group"]

# Color helpers for specific plot types
likert_unsure_colors = ["red", "orange", "lightgray", "lightgreen", "green", "darkgray", "dimgray"]
number_range_colors = ["red", "orange", "lightgreen", "green", "darkgreen"]

# Category ordering for stratified plots
category_orders = {
    "Stakeholder Category": list(reversed([
        "Industry",
        "Industry associations",
        "Civil society",
        "Academia",
        "Public sector",
        "Other",
    ])),
    "Category AB": list(reversed([
        "Category A",
        "Category B",
        "N/A",
    ])),
    "Region": list(reversed([
        "Europe",
        "North America",
        "Asia",
        "LATAM and Caribbean",
        "Oceania",
        "Africa",
        "Middle East",
    ])),
    "SBTi Status": list(reversed([
        "None",
        "NZ only",
        "NT only",
        "Commitment",
        "N/A",
    ])),
    "World Bank Income Group": list(reversed([
        "High income",
        "Upper middle income",
        "Lower middle income",
        "Low income",
    ])),
}

# Optional: sanity check unique values
for col in metadata_columns:
     print(col, "→", df[col].unique())


Region → ['North America' 'Europe' 'Asia' 'LATAM and Caribbean' 'Oceania' 'Africa'
 'Middle East' nan]
SBTi Status → ['n/a - not relevant to me' 'NZ only' 'None' 'NT only' 'N/A' 'Commitment'
 'n/a - not relevant to me.']
Category AB → ['Category A' 'Category B' 'N/A']
Stakeholder Category → ['Academia' 'Industry associations' 'Civil society' 'Other' 'Industry'
 'Public sector' nan]
World Bank Income Group → ['High income' 'Upper middle income' 'Lower middle income' 'Low income'
 nan]


## Step 2: Define functions for plotting

### Sub functions

In [6]:
# ------------------------------------------------------------
# STEP 2: Define functions for plotting
# ------------------------------------------------------------

# ------------------------------------------------------------
# 1. Label utilities (shortening long option texts)
# ------------------------------------------------------------
def is_too_long(option, max_words: int = 4) -> bool:
    """
    Return True if an option label has more than `max_words` words.
    Used to decide when to shorten labels in plots.
    """
    if not isinstance(option, str):
        return False
    return len(option.split()) > max_words


def shorten_label(label, max_words: int = 5):
    """
    Shorten a label to the first `max_words` words and add "..." if truncated.
    """
    if isinstance(label, str):
        words = label.split()
        return " ".join(words[:max_words]) + ("..." if len(words) > max_words else "")
    return label


def shorten_label_2(label, max_words: int = 10):
    """
    Shorten a long label by keeping the first half and the last half of words.
    Example: "This is a very long response label text" → "This is ... label text"
    """
    if isinstance(label, str):
        words = label.split()
        n = len(words)

        if n <= max_words:
            return label

        half = max_words // 2
        start = words[:half]
        end = words[-half:]
        return " ".join(start + ["..."] + end)

    return label


# ------------------------------------------------------------
# 2. Use Question Options to improve ordering
# ------------------------------------------------------------
def improve_order_with_question_options(df_answers: pd.DataFrame, normalize_text):
    """
    Extract and normalize the answer option order from 'Question Options' column,
    using the normalize_text function defined earlier.

    Returns:
        A list of normalized option strings, or an empty list if not available.
    """
    import ast

    def parse_and_normalize(option_str):
        try:
            options = parse_question_options(opt_string)
            if isinstance(options, list):
                return [normalize_text(str(opt)) for opt in options]
        except Exception:
            pass
        return []

    options_list = df_answers["Question Options"].dropna().unique()
    if len(options_list) > 0:
        return parse_and_normalize(options_list[0])
    else:
        return []


# ------------------------------------------------------------
# 3. Color palette helpers
# ------------------------------------------------------------
def get_color_palette(order, is_likert=True, is_rank=False):
    if is_likert:
        palette = likert_unsure_colors
        return align_likert_palette_to_order(order, palette)
    elif is_rank:
        return number_range_colors[:len(order)]
    else:
        return custom_colors[:len(order)]


# ------------------------------------------------------------
# 4. Detect Likert-style answer sets
# ------------------------------------------------------------
def detect_if_likert(order: List[str]) -> bool:
    """
    Determines whether a given list of answer options likely represents a Likert scale.

    Criteria:
    - Length must be between 3 and 6 inclusive.
    - At least 2 known Likert-related keywords must be found in the options.

    Parameters:
    - order (List[str]): List of normalized answer options

    Returns:
    - bool: True if Likert-style, False otherwise
    """
    if not isinstance(order, list) or not all(isinstance(opt, str) for opt in order):
        return False

    LIKERT_KEYWORDS = [
        "all", "most", "sometimes", "rarely", "unsure", "strongly",
        "somewhat", "neutral", "very", "extremely", "not so", "not",
        "all of the time", "most of the time", "other", "some", "none",
    ]

    # Count how many Likert keywords are found at least once in the options
    keyword_hits = sum(
        any(re.search(fr"\b{kw}\b", opt, flags=re.IGNORECASE) for opt in order)
        for kw in LIKERT_KEYWORDS
    )

    return 3 <= len(order) <= 6 and keyword_hits >= 2


# ------------------------------------------------------------
# (Optional) Detect 1–5 rank scales – currently unused
# Kept here for reference; safe to delete later if not needed.
# ------------------------------------------------------------
# def detect_if_rank(order: List[str]) -> bool:
#     """
#     Determines whether a given list of answer options likely represents a
#     1-to-5 numeric scale (e.g., '1 - Strongly disagree', ..., '5 - Strongly agree').
#
#     Criteria:
#     - List must contain exactly 5 items.
#     - Each item must contain a number from 1 to 5, with no repeats.
#     """
#     if not isinstance(order, list) or len(order) != 5:
#         return False
#
#     found_numbers = set()
#     for opt in order:
#         if not isinstance(opt, str):
#             continue
#         match = re.search(r"\b([1-5])\b", opt)
#         if match:
#             found_numbers.add(int(match.group(1)))
#
#     return found_numbers == {1, 2, 3, 4, 5}


# ------------------------------------------------------------
# 5. Likert-style grouped horizontal bar plot
# ------------------------------------------------------------
def plot_likert_bars(
    ctab: pd.DataFrame,
    col: str,
    title: str,
    colors,
    folder_path: str,
    question_id_str,
    question_id_og_str,
    is_likert: bool = True,
    is_meta: bool = False,
    show_labels: bool = True,
    labels=None,
    save_png: bool = False,
    show_plot: bool = True,
    fname: str | None = None,
    abs_values: bool = False,
):
    """
    Create a horizontal stacked bar chart for Likert-style distributions.

    Parameters:
    - ctab: contingency table (index = groups, columns = answer options)
    - col: grouping column name (used in filename)
    - title: chart title
    - colors: list of colors for answer options
    - folder_path: where to save the PNG (if save_png=True)
    - question_id_str: current Question ID
    - question_id_og_str: original question ID/number
    - abs_values: if True, x-axis shows counts instead of percentages
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot stacked horizontal bars
    ctab.plot(
        kind="barh",
        stacked=True,
        ax=ax,
        color=colors[: len(ctab.columns)],
        edgecolor="black",
    )

    # Legend layout
    handles, legend_labels = ax.get_legend_handles_labels()
    n_cols = len(legend_labels) if max(len(l) for l in legend_labels) <= 25 else 1

    # Compute number of legend rows
    legend_rows = (len(legend_labels) + n_cols - 1) // n_cols
    row_height = 0.075
    legend_y_offset = -row_height * legend_rows

    ax.legend(
        handles=handles,
        labels=[shorten_label_2(l, max_words=12) for l in legend_labels],
        loc=("lower left" if n_cols == 1 else "lower center"),
        bbox_to_anchor=(
            (0, -0.05 + legend_y_offset)
            if n_cols == 1
            else (0.5, -0.1 + legend_y_offset)
        ),
        ncol=n_cols,
        frameon=True,
        fontsize=(10 if n_cols == 1 else 9),
    )

    # X-axis range & label
    if abs_values:
        xlim_max = ctab.sum(axis=1).max()
        ax.set_xlabel("Responses")
    else:
        xlim_max = 100
        ax.set_xlabel("Responses (%)")

    # Data labels
    if show_labels:
        if not abs_values:
            # percentage labels directly from ctab values
            for i, row in enumerate(ctab.index):
                left = 0
                for val in ctab.loc[row]:
                    if val > 2:
                        ax.text(
                            left + val / 2,
                            i,
                            f"{val:.1f}%",
                            va="center",
                            ha="center",
                            fontsize=9,
                            color="white",
                        )
                    left += val
        else:
            # calculate pct from absolute counts
            for i, row in enumerate(ctab.index):
                row_total = ctab.loc[row].sum()
                left = 0
                for val in ctab.loc[row]:
                    if row_total > 0:
                        pct = val / row_total * 100
                        if val > 0.025 * xlim_max:
                            ax.text(
                                left + val / 2,
                                i,
                                f"{pct:.1f}%",
                                va="center",
                                ha="center",
                                fontsize=9,
                                color="white",
                            )
                    left += val

    # Clean spines
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.set_xlim([0, xlim_max])
    if xlim_max == 100:
        ax.set_xticks([0, 50, 100])

    ax.set_title(
        textwrap.fill(title, width=90),
        loc="left",
        fontsize=12,
        weight="bold",
    )

    # Save PNG if requested
    if save_png:
        # base_name = f"Q{question_id_str}"
        OG_name = f"Q{question_id_og_str}"
        group_name = col
        fname = f"{OG_name}_{group_name}_grouped_distribution.png"
        os.makedirs(folder_path, exist_ok=True)
        plt.savefig(
            f"{folder_path}/{fname}",
            bbox_inches="tight",
            pad_inches=0.1,
            dpi=300,
        )
        print(f"Saved PNG to: {folder_path}/{fname}")

    if not show_plot:
        plt.close()


# ------------------------------------------------------------
# 6. Grouped bar plots by metadata (Region, Category, etc.)
# ------------------------------------------------------------
def make_group_plot(
    plot_df: pd.DataFrame,
    grp_col: str,
    folder_path: str,
    drop_opts: list,
    title: str,
    y_label: str,
    option_order: list | None,
    save_png: bool = False,
):
    """
    Create grouped bar charts for multi-option questions, split by a metadata column.

    Parameters:
    - plot_df: long-format dataframe with columns:
        ['Question ID', 'Question OG number', 'Option', 'Percent', 'N', grp_col]
    - grp_col: grouping variable ('Region', 'Category AB', etc.) or 'Overall'
    - folder_path: folder where the PNG will be saved
    - drop_opts: list of options to exclude (case-insensitive match)
    - title: chart title
    - y_label: label for y-axis (not used heavily since we plot percentages)
    - option_order: not used to reorder here (kept for compatibility if needed)
    - save_png: whether to save the chart to disk
    """
    # Remove excluded answer options
    plot_df = plot_df[~plot_df["Option"].str.lower().isin(drop_opts)]

    # Handle grouping order
    if grp_col != "Overall":
        base_order = category_orders[grp_col]
        full_order = [item for item in base_order if item != "Overall"] + ["Overall"]
        plot_df[grp_col + "_cat"] = pd.Categorical(
            plot_df[grp_col],
            categories=full_order[::-1],
            ordered=True,
        )
    else:
        plot_df[grp_col + "_cat"] = plot_df[grp_col]

    plot_df = plot_df.sort_values([grp_col + "_cat", "Question ID"])

    # Unique options in final order
    unique_options = plot_df["Option"].drop_duplicates().tolist()
    if len(custom_colors) < len(unique_options):
        raise ValueError("Not enough custom colors for the number of options")

    option_color_map = dict(zip(unique_options, custom_colors[: len(unique_options)]))
    plot_df["Color"] = plot_df["Option"].map(option_color_map)

    # Decide whether to use short labels
    use_short_labels = any(is_too_long(opt) for opt in unique_options)
    option_label_map = {
        opt: (f"Option {i + 1}" if use_short_labels else opt)
        for i, opt in enumerate(unique_options)
    }
    plot_df["ShortOption"] = plot_df["Option"].map(option_label_map)

    y_max = plot_df["Percent"].max() * 1.2 if plot_df["Percent"].max() != 100 else 100

    # Number of groups (facets)
    group_values = plot_df[grp_col + "_cat"].dropna().unique()
    n_groups = len(group_values)
    hori_fig_size = 12 if n_groups == 1 else 3.5

    fig, axes = plt.subplots(
        1,
        n_groups,
        figsize=(hori_fig_size * n_groups, 6),
        sharey=True,
    )

    if n_groups == 1:
        axes = [axes]

    for ax, group in zip(axes, group_values):
        group_df = plot_df[plot_df[grp_col + "_cat"] == group]
        N_label = group_df["N"].unique()[0]

        # Draw bars
        for _, row in group_df.iterrows():
            ax.bar(
                x=row["ShortOption"],
                height=row["Percent"],
                color=row["Color"],
                width=0.6 if n_groups != 1 else 0.8,
            )

            # Value labels
            if row["Percent"] > 2:
                ax.text(
                    x=row["ShortOption"],
                    y=row["Percent"] + 1,
                    s=f"{row['Percent']:.1f}%",
                    ha="center",
                    va="bottom",
                    fontsize=8 if n_groups > 1 else 10,
                )

        ax.set_xlabel(None)
        ax.set_ylim(0, y_max)

        # X labels
        if use_short_labels:
            ax.set_xticks([])
        else:
            ax.set_xticklabels(
                group_df["Option"],
                fontsize=8 if n_groups > 1 else 10,
                rotation=45,
                ha="right",
            )

        if ax == axes[0]:
            ax.set_ylabel("Share of respondents who selected each option / %")
        else:
            ax.set_ylabel("")

        # Group subtitle under plot
        group_label_offset = -0.05 if use_short_labels else -0.20
        ax.text(
            0.5,
            group_label_offset,
            f"{group} (n={N_label})",
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=11,
            fontweight="bold",
        )

    # Legend when using short labels
    if use_short_labels:
        legend_handles = [
            mpatches.Patch(color=option_color_map[full], label=f"{short}: {full}")
            for full, short in option_label_map.items()
        ]
        legend_labels = [
            shorten_label_2(h.get_label(), max_words=16 if n_groups < 3 else 32)
            for h in legend_handles
        ]

        axes[0].legend(
            handles=legend_handles,
            labels=legend_labels,
            loc="upper left",
            bbox_to_anchor=(0, -0.15),
            frameon=True,
            ncol=1,
            borderaxespad=0,
        )

    # Title aligned to first subplot
    ttl = "\n".join(textwrap.wrap(title, 80 if n_groups == 1 else 120))
    left_margin = axes[0].get_position().x0

    fig.suptitle(
        ttl,
        x=left_margin,
        ha="left",
        fontsize=12,
        weight="bold",
    )

    plt.subplots_adjust(wspace=0.2)

    # Save PNG if requested
    if save_png:
        qid_val = plot_df["Question ID"].iloc[0]
        qog_val = plot_df["Question OG number"].iloc[0]
        base_name = f"Q{qid_val}"
        OG_name = f"Q{qog_val}"
        group_name = grp_col.replace(" ", "_").lower()
        fname = f"{base_name}_{group_name}_grouped_distribution_{OG_name}.png"
        os.makedirs(folder_path, exist_ok=True)
        plt.savefig(
            f"{folder_path}/{fname}",
            bbox_inches="tight",
            pad_inches=0.1,
            dpi=300,
        )
        print(f"Saved PNG to: {folder_path}/{fname}")


In [7]:
# ------------------------------------------------------------
# 6. Reorder Colours
# ------------------------------------------------------------
def align_likert_palette_to_order(order, base_colors):
    """
    Ensures likert palette matches semantic direction.
    If order starts with Agree, reverse the palette so Agree becomes green.
    """
    if not order:
        return base_colors

    first = str(order[0]).strip().lower()
    last = str(order[-1]).strip().lower()

    # If scale goes Agree -> Disagree, reverse colors so Agree is green
    if "agree" in first and "disagree" in last:
        return list(reversed(base_colors[:len(order)]))

    # If scale goes Disagree -> Agree, keep as is
    return base_colors[:len(order)]


## Step 3: Define pre-process answers

In [8]:
# ============================================================
# STEP 3: PRE-PROCESS ANSWERS PER QUESTION (VALIDATE VS OPTIONS, TAG 'OTHER')
# ============================================================

# This will collect all open-ended / unmatched answers across questions
open_ended_answers = pd.DataFrame(columns=["Question ID", "Raw Response"])


# ------------------------------------------------------------
# Helper: remove dots (.) from non-numeric strings
# ------------------------------------------------------------
def remove_dots_if_not_numeric(x):
    """
    If x is a string and NOT a valid integer/float, remove '.' characters.
    Keeps '.' if the value looks like a number (e.g. '1.0', '2.5').
    """
    if isinstance(x, str) and not re.fullmatch(r"\d+(\.\d+)?", x):
        return re.sub(r"\.", "", x)
    return x


# ------------------------------------------------------------
# Core: preprocess answers for a single question
# ------------------------------------------------------------
def preprocess_answers(df_q: pd.DataFrame, qid):
    """
    Clean and validate answers for a single question (Question ID = qid).

    Steps:
      1. Normalize 'Answer' (strip, lowercase, remove noisy patterns, normalize_text).
      2. Normalize the 'Question Options' list to a comparable form.
      3. Mark answers as valid if they match one of the normalized options
         or equal "Not relevant to me".
      4. Collect all non-matching answers in a global `open_ended_answers` table.
      5. Tag non-matching answers as 'Other' in df_q.

    Returns:
      df_q with cleaned 'Answer' and without helper columns.
    """
    global open_ended_answers

    df_q = df_q.copy()

    # ----------------------------
    # 1. Normalize the Answer text
    # ----------------------------
    df_q["Answer"] = df_q["Answer"].astype(str).str.strip().str.lower()
    # -------------------------------
    # EXTRA NORMALIZATION (Select-all + Rank)
    # -------------------------------
    # 1) Standardize boolean variants -> "True"/"False"
    bool_map = {
        "true": "True", "false": "False",
        "1": "True", "0": "False",
        "yes": "True", "no": "False"
    }
    df_q["Answer"] = df_q["Answer"].replace(bool_map)

    # 2) If rank-like numeric, keep as clean digit string (e.g., "1", "2", "3", "4")
    # (avoids issues like "1.0", " 2 ", etc.)
    df_q["Answer"] = df_q["Answer"].apply(lambda x: str(int(float(x))) if re.fullmatch(r"\d+(\.\d+)?", str(x)) else x)

    df_q["Answer"] = df_q["Answer"].str.replace(r"\s+", " ", regex=True)
    df_q["Answer"] = df_q["Answer"].str.replace("’", "'", regex=False)  # malformed apostrophe
    df_q["Answer"] = df_q["Answer"].apply(remove_dots_if_not_numeric)

    # Keep only the "option N" code if the answer starts like "option N: ..."
    df_q["Answer"] = df_q["Answer"].apply(
        lambda x: re.sub(r"^(option \d+):.*", r"\1", x.strip())
    )

    # Apply higher-level normalization (from earlier cell)
    df_q["Answer"] = df_q["Answer"].apply(normalize_text)

    

    def canon(x):
        """Canonical form used ONLY for matching answers to options."""
        if pd.isna(x):
            return ""
        s = str(x).strip().lower()
        s = s.replace("’", "'")
        s = re.sub(r"\s+", " ", s)

        # remove dots ONLY if not numeric (match your answer logic)
        if not re.fullmatch(r"\d+(\.\d+)?", s):
            s = s.replace(".", "")

        # strip "Option X:" prefixes if they appear
        s = re.sub(r"^(option\s*\d+)\s*:\s*.*", r"\1", s).strip()

        # remove plus signs like your options cleaner
        s = re.sub(r"\+", "", s).strip()

        return s

    def parse_question_options(opt_string):
        """
        Robust parser:
        - tries ast.literal_eval
        - returns [] if invalid
        """
        if pd.isna(opt_string):
            return []
        raw = str(opt_string).strip().replace("’", "'")

        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return []

    # --- Build normalized options list per row ---
    df_q["Normalized Options"] = df_q["Question Options"].apply(parse_question_options)

    # --- Canonical answer for matching ---
    df_q["_ans_canon"] = df_q["Answer"].apply(canon)

    def is_valid_row(row):
        opts = row["Normalized Options"]

        # IMPORTANT:
        # If Question Options are invalid/unparseable, DO NOT force "Other"
        # (this prevents the "everything becomes Other" failure mode)
        if not isinstance(opts, list) or len(opts) == 0:
            return True

        opts_canon = [canon(o) for o in opts]
        ans = row["_ans_canon"]

        return (ans in opts_canon) or (ans in {"not relevant to me", "not applicable to me"})

    df_q["Is Valid"] = df_q.apply(is_valid_row, axis=1)

    # --- Anything that failed matching becomes "Other" ---
    mask_other = ~df_q["Is Valid"]

    if mask_other.any():
        new_open_ended = pd.DataFrame({
            "Question ID": [qid] * int(mask_other.sum()),
            "Raw Response": df_q.loc[mask_other, "Answer"]
        })
        open_ended_answers = pd.concat([open_ended_answers, new_open_ended], ignore_index=True)

    df_q.loc[mask_other, "Answer"] = "Other"

    # cleanup
    df_q.drop(columns=["Normalized Options", "Is Valid", "_ans_canon"], inplace=True, errors="ignore")


    
    # ----------------------------
    # Tag unmatched responses as 'Other'
    # ----------------------------
    df_q.loc[mask_other, "Answer"] = "Other"

    # Drop helper columns before returning
    return df_q.drop(columns=["Normalized Options", "Is Valid"])


# ------------------------------------------------------------
# Debug utility: print parsed Question Options
# ------------------------------------------------------------
def print_question_options(df_q: pd.DataFrame):
    """
    Print out the parsed 'Question Options' for debugging.
    """
    unique_options = df_q["Question Options"].dropna().unique()

    for i, opt_string in enumerate(unique_options, start=1):
        parsed = parse_question_options(opt_string)
        if parsed:
            print(f"Question Options {i}: {parsed}")
        else:
            print(f"Question Options {i}: ⚠️ Invalid format → {opt_string}")

## Step 4: Define show answer distribution 

In [10]:
# ============================================================
# STEP 4: SHOW ANSWER DISTRIBUTION FOR A SINGLE QUESTION
# ============================================================

def show_answer_distribution(df_q, save_csv=False, csv_path=None):
    """
    Compute and display the distribution of answers for a specific question.

    Parameters
    ----------
    df_q : pd.DataFrame
        Filtered DataFrame containing rows for ONE question only.
    save_csv : bool
        If True, the distribution table is saved as a CSV.
    csv_path : str or None
        Optional custom save path. If None, an automatic filename is generated.

    Returns
    -------
    pd.DataFrame
        A formatted distribution table:
        [Question ID | OG Question Number | Question | Answer | Count | Percentage]
    """

    # ------------------------------------------------------------
    # Extract question metadata
    # ------------------------------------------------------------
    question_text = df_q["Question"].dropna().unique()
    question_id = df_q["Question ID"].dropna().unique()
    question_id_og = df_q["OG Question Number"].dropna().unique()

    question_str = question_text[0] if len(question_text) > 0 else "Unknown"
    question_id_str = question_id[0] if len(question_id) > 0 else "Unknown"
    question_id_og_str = question_id_og[0] if len(question_id_og) > 0 else "Unknown"

    # ------------------------------------------------------------
    # Compute distribution
    # ------------------------------------------------------------
    counts = df_q["Answer"].value_counts(normalize=False).reset_index()
    counts.columns = ["Answer", "Count"]

    counts["Percentage"] = round(
        counts["Count"] / counts["Count"].sum() * 100,
        1
    )

    # Attach question metadata to table
    counts["Question"] = question_str
    counts["Question ID"] = question_id_str
    counts["Question OG number"] = question_id_og_str

    # Keep consistent column order
    counts = counts[
        ["Question ID", "Question OG number", "Question", "Answer", "Count", "Percentage"]
    ]

    # ------------------------------------------------------------
    # Save output if requested
    # ------------------------------------------------------------
    if save_csv:
        if csv_path is None:
            base_name = f"Q{question_id_str}"
            OG_name = f"Q{question_id_og_str}"
            cleaned_col = "overall"
            csv_path = f"Q{OG_name}_{cleaned_col}_distribution_{base_name}.csv"

        out_path = f"../output/quanti_results/{csv_path}"
        counts.to_csv(out_path, index=False)
        print(f"Saved distribution to: {out_path}")

    return counts


## Step 5: Define show metadata breakdown

In [11]:
# ===========================================================================
# STEP 5: SHOW METADATA BREAKDOWN (OVERALL + BY REGION / WB / CATEGORY / STAKEHOLDER)
# ===========================================================================

def show_metadata_breakdown(
    df_q,
    show_labels=True,
    plot_overall_only=False,
    save_csv=False,
    csv_path=None,
    save_png=False,
    save_excel=False,
    show_plot=True,          # currently passed through to plot_likert_bars
    only_stake_category=True,  # legacy flag – only used if 'Stakeholder Category' exists
    output_folder="../output/quanti_result/",
):
    """
    For a single question (df_q), compute and plot distributions:
      - Overall answer distribution (percentage & counts)
      - By metadata columns (Region, SBTi Status, Category AB, Stakeholder Group, WB Income Group)
    And optionally:
      - Save CSVs
      - Save Excel file with all tables
      - Save PNG charts

    Returns:
      overall_ctab_ref          (1-row DF, % distribution, labelled for plotting)
      overall_ctab_absolute_ref (1-row DF, count distribution, labelled for plotting)
      is_likert                 (bool flag: does this look like a Likert question?)
      order                     (list of answer category order)
      colors                    (list of colors used)
      folder_path               (folder where outputs are written)
    """

    # ------------------------------------------------------------
    # 0. Create per-question-group folder for outputs
    # ------------------------------------------------------------
    question_group_val = df_q["Question Group"].iloc[0]        # Replaced "Mapping Group" with "Question Group"
    safe_folder_name = str(question_group_val).strip().replace(",", "_").replace("/", "_")
    folder_path = os.path.join(output_folder, f"Question_Group_{safe_folder_name}")
    os.makedirs(folder_path, exist_ok=True)

    # ------------------------------------------------------------
    # 1. Extract basic question metadata
    # ------------------------------------------------------------
    question_text = df_q["Question"].dropna().unique()
    question_id = df_q["Question ID"].dropna().unique()
    question_id_og = df_q["OG Question Number"].dropna().unique()

    question_str = question_text[0] if len(question_text) > 0 else "Unknown"
    question_id_str = (
        str(int(question_id[0])) if len(question_id) > 0 else "Unknown"
    )
    question_id_og_str = (
        str(int(question_id_og[0])) if len(question_id_og) > 0 else "Unknown"
    )

    # ------------------------------------------------------------
    # 2. Filter out irrelevant answers (e.g. 'Not relevant to me')
    # ------------------------------------------------------------
    irrelevant_answers = ["not relevant to me"]
    df_filtered_all = df_q[
        ~df_q["Answer"].astype(str).str.lower().isin(irrelevant_answers)
    ].copy()

    # ------------------------------------------------------------
    # 3. Determine answer order (from Question Options → fallback to freq)
    # ------------------------------------------------------------
    order = improve_order_with_question_options(df_filtered_all, normalize_text)
    if not order:
        order = df_filtered_all["Answer"].value_counts().index.tolist()

    # Likert-flag detection
    is_likert = detect_if_likert(order)

    # Color palette for this question
    colors = get_color_palette(order, is_likert=is_likert)

    # Where to store Excel sheets
    excel_sheets = {}

    # ------------------------------------------------------------
    # 4. OVERALL distribution (percent & counts)
    # ------------------------------------------------------------
    overall_ctab = (
        df_filtered_all["Answer"]
        .value_counts(normalize=True)
        .reindex(order)
        .fillna(0)
        * 100
    ).to_frame().T

    overall_ctab.index = [f"Overall (n={len(df_filtered_all)})"]

    # “Ref” copy with nicer row label for plotting alongside other groups
    overall_ctab_ref = overall_ctab.copy()

    # Extract option phrase from question text after "|"
    question_option_str = df_filtered_all["Question"].iloc[0].split("|")[-1].strip()
    if "claims" in question_option_str:
        # Remove everything after the first '(' if present
        question_option_str = question_option_str.split("(")[0].strip()

    question_option_str_full = f"Q{question_id_str} {question_option_str} (n={len(df_filtered_all)})"
    words = question_option_str_full.split()
    question_option_str_full = "\n".join(
        [" ".join(words[i : i + 4]) for i in range(0, len(words), 4)]
    )
    overall_ctab_ref.index = [question_option_str_full]

    # Absolute counts
    overall_ctab_absolute = (
        df_filtered_all["Answer"]
        .value_counts()
        .reindex(order)
        .fillna(0)
        .to_frame()
        .T
    )

    overall_ctab_absolute_ref = overall_ctab_absolute.copy()
    overall_ctab_absolute_ref.index = [question_option_str_full]

    # Add metadata columns for Excel export
    overall_ctab_absolute.insert(0, "Question", question_str)
    overall_ctab_absolute.insert(0, "Question ID", question_id_str)
    overall_ctab_absolute.insert(0, "Question OG number", question_id_og_str)

    excel_sheets["Overall"] = overall_ctab_absolute
    excel_sheets["Overall_%"] = overall_ctab

    # ------------------------------------------------------------
    # 5. Overall-only chart + CSV (no metadata breakdown)
    # ------------------------------------------------------------
    if not only_stake_category:
        # Single overall plot (no by-metadata splits)
        plot_likert_bars(
            overall_ctab,
            col="overall",
            title=f"Q{question_id_og_str}: {question_str}",
            colors=colors,
            folder_path=folder_path,
            question_id_str=question_id_str,
            question_id_og_str=question_id_og_str,
            is_likert=is_likert,
            is_meta=True,
            show_labels=show_labels,
            labels=[shorten_label_2(l) for l in order],
            save_png=save_png,
            fname=(
                f"Q{question_id_og_str}_overall_distribution_Q{question_id_str}"
                if save_png
                else None
            ),
        )

    if save_csv:
        dist_csv = csv_path or f"Q{question_id_og_str}_overall_answer_distribution_Q{question_id_str}.csv"
        overall_ctab.to_csv(f"{folder_path}/{dist_csv}")
        print(f"Saved CSV to: {dist_csv}")

    # ------------------------------------------------------------
    # 6. BY METADATA (Region, Category AB, SBTi Status, Stakeholder Category, WB Income Group)
    # ------------------------------------------------------------
    for col in metadata_columns:
        df_filtered = df_filtered_all.copy()

        # Absolute counts by group
        ctab_absolute = (
            pd.crosstab(df_filtered[col], df_filtered["Answer"], margins=False)
            .fillna(0)
        )

        # Reindex groups according to category_orders
        ctab_absolute = ctab_absolute.reindex(category_orders[col])

        # Ensure answer columns follow 'order'
        ctab_absolute = ctab_absolute.T.reindex(order, fill_value=0).T

        # Add question metadata
        ctab_absolute.reset_index(inplace=True)
        ctab_absolute.insert(0, "Question", question_str)
        ctab_absolute.insert(0, "Question ID", question_id_str)
        ctab_absolute.insert(0, "Question OG number", question_id_og_str)

        # Row-normalized percentages
        ctab = (
            pd.crosstab(df_filtered[col], df_filtered["Answer"], normalize="index")
            * 100
        )
        ctab = ctab.reindex(category_orders[col]).dropna(how="all")
        ctab = ctab.T.reindex(order, fill_value=0).T

        # n per group for label
        group_counts = df_filtered[col].value_counts().to_dict()
        ctab.index = [
            f"{label} (n={group_counts.get(label, 0)})" for label in ctab.index
        ]

        # Add overall row to % ctab (so overall appears in same chart)
        if len(ctab) > 1:
            ctab = pd.concat([ctab, overall_ctab], axis=0)

        # Build a matching "Overall" row for the absolute crosstab
        ctab_absolute_merge = overall_ctab_absolute.copy()
        ctab_absolute_merge[col] = np.nan
        # Reorder columns to match existing ctab_absolute
        ctab_absolute_merge = ctab_absolute_merge[ctab_absolute.columns]
        # Join
        ctab_absolute = pd.concat([ctab_absolute, ctab_absolute_merge], axis=0)
        ctab_absolute.fillna({col: "Overall"}, inplace=True)
        ctab_absolute.reset_index(drop=True, inplace=True)

        # Ensure the final ctab has columns in 'order'
        ctab = ctab[ctab.columns.intersection(order)].reindex(columns=order)

        cleaned_col = col.replace(" ", "_").lower().replace("/", "_")

        # Add to Excel export dictionary
        excel_sheets[col] = ctab_absolute
        excel_sheets[col + "_%"] = ctab

        # Plot by group unless we only want overall OR only stakeholder category
        if not plot_overall_only:
            if only_stake_category and col != "Stakeholder Category":
                # Legacy: when you only want 'Stakeholder Category' plots
                continue

            plot_likert_bars(
                ctab,
                col=cleaned_col,
                title=f"Q{question_id_og_str}: {question_str} - by {col}",
                colors=colors,
                folder_path=folder_path,
                question_id_str=question_id_str,
                question_id_og_str=question_id_og_str,
                is_likert=is_likert,
                is_meta=True,
                show_labels=show_labels,
                labels=order,
                save_png=save_png,
                fname=(
                    f"Q{question_id_og_str}_{cleaned_col}_grouped_distribution_Q{question_id_str}"
                    if save_png
                    else None
                ),
            )

        if save_csv:
            dist_csv = (
                f"{question_id_og_str}_{cleaned_col}_answer_distribution_Q{question_id_str}.csv"
            )
            ctab_absolute.to_csv(f"{folder_path}/{dist_csv}", index=False)
            print(f"Saved CSV to: {dist_csv}")

    # ------------------------------------------------------------
    # 7. OPTIONAL: Save all distributions to one Excel file
    # ------------------------------------------------------------
    if save_excel:
        excel_path = (
            f"{folder_path}/Q{question_id_og_str}_all_distributions_Q{question_id_str}.xlsx"
        )
        with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
            for sheet_name, df_sheet in excel_sheets.items():
                df_sheet.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        print(f"Saved all distributions to: {excel_path}")

    # Debug: inspect last ctab_absolute + merge
    display(ctab_absolute)
    print(ctab_absolute.index)
    display(ctab_absolute_merge)
    print(ctab_absolute_merge.index)

    return (
        overall_ctab_ref,
        overall_ctab_absolute_ref,
        is_likert,
        order,
        colors,
        folder_path,
    )


## Step 6: Define plot grouped lykert

In [12]:
# ============================================================
# STEP 6: PLOT GROUPED LIKERT (BLOCKS OF TRUE/FALSE QUESTIONS)
# ============================================================

def plot_grouped_likert(
    df,
    prefix_string,
    show_labels: bool = True,
    overall_only: bool = False,
    group_cols: list | None = None,
    save_csv: bool = False,
    save_png: bool = False,
    save_excel: bool = False,
    output_folder: str = "../output/quanti_results_v2/",
):
    """
    Plot grouped horizontal bar charts for blocks of TRUE/FALSE questions
    (e.g. where each sub-question starts with a common prefix_string).

    For each answer option (the text after the "|" in the Question column):
      - Compute % of respondents answering TRUE across all included questions
      - Plot Overall distribution across options
      - Optionally break down by:
          * Stakeholder Category
          * Region
          * SBTi Target
          * Category AB
          * World Bank Income Group

    Saves:
      - PNGs for Overall and each grouping (if save_png=True)
      - CSV files with grouped distributions (if save_csv=True)
      - One Excel file per mapping group with all sheets (if save_excel=True)

    Parameters
    ----------
    df : pd.DataFrame
        Full quantitative dataset (long format, one row per respondent-answer).
    prefix_string : str
        Text prefix of the question block to analyse
        (e.g. "To what extent do you agree with the following statements").
    show_labels : bool
        Whether to show value labels on bars (passed down to make_group_plot).
    overall_only : bool
        If True, only overall distribution is plotted (no group breakdowns).
    group_cols : list | None
        Ignored in original logic; grouping columns are fixed inside function.
    save_csv : bool
        Save grouped distributions as CSV.
    save_png : bool
        Save plots as PNG.
    save_excel : bool
        Save all distributions for this block as a single Excel workbook.
    output_folder : str
        Base output folder for all generated files.
    """

    # --------------------------------------------------------
    # 0. Create folder per Question Group
    # --------------------------------------------------------
    question_group_val = df["Question Group"].iloc[0]   # Replaced "Mapping Group" with "Question Group"
    OG_question_val = str(df["OG Question Number"].iloc[0])

    safe_folder_name = str(question_group_val).strip().replace(",", "_").replace("/", "_")
    folder_path = os.path.join(output_folder, f"Question_Group_{safe_folder_name}")
    os.makedirs(folder_path, exist_ok=True)

    # NOTE: in original code, group_cols parameter is ignored and replaced:
    group_cols = [
        "Stakeholder Category",
        "Region",
        "SBTi Target",
        "Category AB",
        "World Bank Income Group",
    ]

    # --------------------------------------------------------
    # 1. Filter dataframe to the block of questions with prefix_string
    # --------------------------------------------------------
    df_block = df[df["Question"].str.startswith(prefix_string)].copy()
    if df_block.empty:
        print(f"No questions found with prefix: {prefix_string}")
        return

    # Extract the answer "option" from the part after the "|" in Question
    df_block["Option"] = (
        df_block["Question"]
        .str.extract(r"\|\s*(.*)$")[0]
        .fillna(df_block["Question"])
    )

    # Options we don't want to show
    drop_opts = {"not relevant to me", "not applicable to me"}

    # --------------------------------------------------------
    # 2. Compute overall share TRUE per option to determine order
    # --------------------------------------------------------
    option_totals = {}

    for option, opt_df in df_block.groupby("Option"):
        if option.lower().strip() in drop_opts:
            continue

        true_tot, tot = 0, 0
        for qid in opt_df["Question ID"].dropna().unique():
            sub = preprocess_answers(df[df["Question ID"] == qid], qid=qid)
            true_tot += (sub["Answer"] == "True").sum()
            tot += sub["Answer"].notna().sum()

        if tot:
            option_totals[option] = true_tot / tot

    # Sort options from highest % TRUE to lowest
    option_order = sorted(option_totals, key=option_totals.get, reverse=True)

    # --------------------------------------------------------
    # 3. Build OVERALL distribution dataframe
    # --------------------------------------------------------
    rows = []
    for option, opt_df in df_block.groupby("Option"):
        true_tot, tot = 0, 0
        for qid in opt_df["Question ID"].dropna().unique():
            sub = preprocess_answers(df[df["Question ID"] == qid], qid=qid)
            true_tot += (sub["Answer"] == "True").sum()
            tot += sub["Answer"].notna().sum()

        if tot:
            rows.append(
                {
                    "Question ID": qid,
                    "Question OG number": (
                        opt_df["OG Question Number"].values[0]
                        if "OG Question Number" in opt_df.columns
                        else ""
                    ),
                    "Question": opt_df["Question"].values[0],
                    "Overall": "Overall",
                    "Option": option,
                    "Percent": round(true_tot / tot * 100, 1),
                    "N": tot,
                }
            )

    if not rows:
        print("No valid responses.")
        return

    plot_df = pd.DataFrame(rows)
    plot_df_overall = plot_df.copy()

    # Excel sheets dictionary (each key → a sheet in final Excel)
    excel_sheets = {}
    excel_sheets["Overall"] = plot_df_overall

    # Build a compact question ID group string like "231, 232, 233"
    question_ID_group = ", ".join(
        map(str, np.sort(list(map(int, plot_df["Question ID"].dropna().unique()))))
    )
    N_number = str(plot_df["N"][0])

    # --------------------------------------------------------
    # 4. Plot OVERALL distribution (no metadata split)
    # --------------------------------------------------------
    make_group_plot(
        plot_df,
        grp_col="Overall",
        folder_path=folder_path,
        drop_opts=drop_opts,
        title=f"Q{question_ID_group}: {prefix_string} (n={N_number}) – Overall distribution",
        y_label="Overall",
        option_order=option_order,
        save_png=save_png,
    )

    if save_csv:
        qid_val = plot_df["Question ID"].iloc[0]
        qog_val = plot_df["Question OG number"].iloc[0]
        base_name = f"Q{qid_val}"
        OG_name = f"Q{qog_val}"
        fname = f"{OG_name}_overall_grouped_distribution_{base_name}.csv"
        plot_df.to_csv(f"{folder_path}/{fname}", index=False)
        print(f"Saved grouped distribution to: {fname}")

    # --------------------------------------------------------
    # 5. Plot distributions BY GROUP (Stakeholder / Region / etc.)
    # --------------------------------------------------------
    if not overall_only:
        for grp_col in group_cols:
            print(f"\n\033[1mProcessing {prefix_string} - by {grp_col}\033[0m")

            rows = []
            for option, opt_df in df_block.groupby("Option"):
                for qid in opt_df["Question ID"].dropna().unique():
                    sub = df[(df["Question ID"] == qid) & df[grp_col].notna()].copy()
                    sub = preprocess_answers(sub, qid=qid)

                    for val, g in sub.groupby(grp_col):
                        tot = g["Answer"].notna().sum()
                        true_ = (g["Answer"] == "True").sum()
                        if tot:
                            rows.append(
                                {
                                    grp_col: val,
                                    "Option": option,
                                    "Percent": round(true_ / tot * 100, 1),
                                    "N": tot,
                                    "Question": opt_df["Question"].values[0],
                                    "Question ID": qid,
                                    "Question OG number": (
                                        opt_df["OG Question Number"].values[0]
                                        if "OG Question Number" in opt_df.columns
                                        else ""
                                    ),
                                }
                            )

            if not rows:
                print(f"No valid responses for {grp_col}.")
                continue

            plot_df = pd.DataFrame(rows)

            # Add Overall rows to this grouped dataframe so
            # they appear on the same chart as the categories
            plot_df_overall_merge = plot_df_overall.copy()
            plot_df_overall_merge = plot_df_overall_merge.rename(
                columns={"Overall": grp_col}
            )

            if len(plot_df[grp_col].unique()) > 1:
                plot_df = pd.concat([plot_df_overall_merge, plot_df], axis=0)

            excel_sheets[grp_col] = plot_df

            # Compute label strings for title
            question_ID_group = ", ".join(
                map(
                    str,
                    np.sort(
                        list(map(int, plot_df["Question ID"].dropna().unique()))
                    ),
                )
            )
            N_number = str(plot_df["N"].unique().max())

            # Grouped plot by grp_col
            make_group_plot(
                plot_df,
                grp_col=grp_col,
                folder_path=folder_path,
                drop_opts=drop_opts,
                title=(
                    f"Q{question_ID_group}: {prefix_string} "
                    f"(n={N_number}) – by {grp_col}"
                ),
                y_label=grp_col,
                option_order=option_order,
                save_png=save_png,
            )

            if save_csv:
                qid_val = plot_df["Question ID"].iloc[0]
                qog_val = plot_df["Question OG number"].iloc[0]
                base_name = f"Q{qid_val}"
                OG_name = f"Q{qog_val}"
                group_name = grp_col.replace(" ", "_").lower()
                fname = f"Q{OG_name}_{group_name}_grouped_distribution_{base_name}.csv"
                plot_df.to_csv(f"{folder_path}/{fname}", index=False)
                print(f"Saved grouped distribution to: {fname}")

        # --------------------------------------------------------
        # 6. Save one Excel workbook with all sheets for this block
        # --------------------------------------------------------
        if save_excel:
            excel_path = (
                f"{folder_path}/QG_{safe_folder_name}_all_distributions_Q{OG_question_val}.xlsx"
            )
            with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
                for sheet_name, df_sheet in excel_sheets.items():
                    df_sheet.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            print(f"Saved all distributions to: {excel_path}")

    return plot_df


## Step 7: Define plot grouped rank

In [13]:
# ============================================================
# STEP 7: PLOT GROUPED RANK / BOOLEAN (BLOCK QUESTIONS)
# ============================================================

def plot_grouped_rank(
    df,
    prefix_string,
    show_labels: bool = True,
    overall_only: bool = False,
    group_cols: list | None = None,
    save_csv: bool = False,
    save_png: bool = False,
    save_excel: bool = False,
):
    """
    Plot grouped horizontal bar charts for a block of ranking / boolean questions
    whose text starts with `prefix_string`.

    Supports two modes (auto-detected from Answer column):
      - "ranked": Answer ∈ {1, 2, 3, 4, 5}
      - "boolean": Answer ∈ {"True", "False"}

    For each option (sub-question after "|"):
      * Overall distribution
      * Optional breakdown by:
          - Stakeholder Category
          - Region
          - SBTi Target
          - Category AB
          - World Bank Income Group

    Saves:
      * PNGs of charts (if save_png=True)
      * CSVs of distributions (if save_csv=True)
      * One Excel workbook per question group with all sheets (if save_excel=True)
    """

    # --------------------------------------------------------
    # 0. Detect whether question is boolean or ranked (1–5)
    # --------------------------------------------------------
    def detect_mode(df_sub):
        """Return 'ranked' (1–5), 'boolean' (True/False), or raise error."""
        answers = df_sub["Answer"].dropna().unique()
        display(answers)

        # Try numeric 1–5
        try:
            numeric_vals = [int(a) for a in answers]
            if all(1 <= val <= 5 for val in numeric_vals):
                return "ranked"
        except Exception:
            pass

        # Try boolean
        if set(answers).issubset({"True", "False"}):
            return "boolean"

        raise ValueError(
            "Unrecognized Answer format. Only 'True/False' or numeric ranks 1–5 are supported."
        )

    # --------------------------------------------------------
    # 1. Prepare folder for this Question Group
    # --------------------------------------------------------
    question_group_val = df["Question Group"].iloc[0]   # Replaced "Mapping Group" with "Question Group"
    OG_question_val = str(df["OG Question Number"].iloc[0])

    safe_folder_name = str(question_group_val).strip().replace(",", "_").replace("/", "_")
    folder_path = os.path.join("../output/quanti_results/", f"Question_Group_{safe_folder_name}")
    os.makedirs(folder_path, exist_ok=True)

    if group_cols is None:
        group_cols = [
            "Stakeholder Category",
            "Region",
            "SBTi Target",
            "Category AB",
            "World Bank Income Group",
        ]

    # --------------------------------------------------------
    # 2. Filter to block of questions that share the prefix
    # --------------------------------------------------------
    df_block = df[df["Question"].str.startswith(prefix_string)].copy()
    if df_block.empty:
        print(f"No questions found with prefix: {prefix_string}")
        return

    # Extract sub-option text (after "|") for each question
    df_block["Option"] = (
        df_block["Question"]
        .str.extract(r"\|\s*(.*)$")[0]
        .fillna(df_block["Question"])
    )

    # Options to exclude
    drop_opts = {"not relevant to me", "not applicable to me"}

    # Mode: "ranked" or "boolean"
    mode = detect_mode(df_block)
    rank_order = ["1", "2", "3", "4", "5"]
    excel_sheets = {}

    # --------------------------------------------------------
    # 3. Inner helper: plot a single grouped chart
    # --------------------------------------------------------
    def _make_plot(
        plot_df,
        grp_col,
        folder_path,
        title,
        y_label,
        hue_order,
        hue_label,
        save_png=False,
    ):
        """
        plot_df must have:
          - y_label column (e.g. 'Overall' or a group column)
          - 'Percent' (boolean mode) OR 'Count' (ranked mode; used as Percent)
          - hue_label column: 'Rank' (ranked) or 'Option' (boolean)
        """
        fig, ax = plt.subplots(figsize=(10, 6))

        sns.barplot(
            data=plot_df,
            x="Percent",         # in ranked mode we use Percent too (see generator)
            y=y_label,
            hue=hue_label,
            hue_order=hue_order,
            orient="h",
            ax=ax,
        )

        if show_labels:
            for p in ax.patches:
                width = p.get_width()
                if width > 3:
                    ycent = p.get_y() + p.get_height() / 2
                    ax.text(
                        width - 1,
                        ycent,
                        f"{width:.1f}%",
                        ha="right",
                        va="center",
                        color="white",
                        fontsize=7,
                    )

        ax.set_xlim(0, 100)
        ax.set_xlabel("Share of responses/%")
        ax.set_ylabel(None)

        ttl = "\n".join(textwrap.wrap(title, 80))
        ax.set_title(ttl, loc="left", fontsize=12, weight="bold", pad=0)

        ax.legend(
            title=hue_label,
            loc="lower left",
            bbox_to_anchor=(0, -0.5),
            ncol=5,
            fontsize=8,
        )

        fig.set_figheight(5.5)
        plt.subplots_adjust(bottom=0.2)
        plt.show()

        if save_png:
            qid_val = plot_df["Question ID"].iloc[0]
            qog_val = plot_df["Question OG number"].iloc[0]
            base_name = f"Q{qid_val}"
            OG_name = f"Q{qog_val}"
            group_name = grp_col.replace(" ", "_").lower()
            fname = f"{base_name}_{group_name}_grouped_distribution_{OG_name}.png"
            plt.savefig(f"{folder_path}/{fname}", bbox_inches="tight", pad_inches=0.1, dpi=300)
            print(f"Saved PNG to: {folder_path}/{fname}")
            plt.close()

    # --------------------------------------------------------
    # 4. Inner helper: build distribution table (overall or grouped)
    # --------------------------------------------------------
    def generate_distribution(df_block, grp_col=None):
        """
        Returns a DataFrame with either:
         - boolean mode: cols [grp_col/Overall, Option, Percent, N, Question, QID, OGQ]
         - ranked mode:  cols [grp_col/Overall, Option, Rank, Count(=%), N, Question, QID, OGQ]
        """
        rows = []

        for option, opt_df in df_block.groupby("Option"):
            if option.lower().strip() in drop_opts:
                continue

            for qid in opt_df["Question ID"].dropna().unique():
                sub = df[df["Question ID"] == qid].copy()

                if grp_col:
                    sub = sub[sub[grp_col].notna()]
                    group_vals = sub[grp_col].unique()
                else:
                    group_vals = ["Overall"]

                sub = preprocess_answers(sub, qid=qid)

                for val in group_vals:
                    sub_group = sub if grp_col is None else sub[sub[grp_col] == val]
                    total = sub_group["Answer"].notna().sum()
                    if not total:
                        continue

                    if mode == "boolean":
                        true_count = (sub_group["Answer"] == "True").sum()
                        rows.append(
                            {
                                (grp_col if grp_col else "Overall"): val,
                                "Option": option,
                                "Percent": round(true_count / total * 100, 1),
                                "N": total,
                                "Question": opt_df["Question"].values[0],
                                "Question ID": qid,
                                "Question OG number": opt_df["OG Question Number"].values[0],
                            }
                        )

                    elif mode == "ranked":
                        for rank in rank_order:
                            rank_count = (sub_group["Answer"] == rank).sum()
                            rows.append(
                                {
                                    (grp_col if grp_col else "Overall"): val,
                                    "Option": option,
                                    "Rank": rank,
                                    "Count": round(rank_count / total * 100, 1),
                                    "N": total,
                                    "Question": opt_df["Question"].values[0],
                                    "Question ID": qid,
                                    "Question OG number": opt_df["OG Question Number"].values[0],
                                }
                            )

        return pd.DataFrame(rows)

    # ========================================================
    # 5. OVERALL PLOT
    # ========================================================
    plot_df = generate_distribution(df_block)
    plot_df_overall = plot_df.copy()

    # Decide whether hue is Rank (ranked mode) or Option (boolean mode)
    hue_label = "Rank" if mode == "ranked" else "Option"
    hue_order = rank_order if mode == "ranked" else sorted(plot_df["Option"].unique())

    question_ID_group = ", ".join(
        map(str, np.sort(plot_df["Question ID"].dropna().unique()))
    )
    N_number = str(plot_df["N"].max())

    # In ranked mode, we still use 'Percent' as value to plot
    if mode == "ranked":
        plot_df = plot_df.rename(columns={"Count": "Percent"})

    _make_plot(
        plot_df,
        grp_col="Overall",
        folder_path=folder_path,
        title=f"Q{question_ID_group}: {prefix_string} (n={N_number}) – Overall distribution",
        y_label="Overall",
        hue_order=hue_order,
        hue_label=hue_label,
        save_png=save_png,
    )

    excel_sheets["Overall"] = plot_df.copy()

    if save_csv:
        fname = f"Q{OG_question_val}_overall_distribution.csv"
        plot_df.to_csv(f"{folder_path}/{fname}", index=False)
        print(f"Saved overall distribution to: {fname}")

    # ========================================================
    # 6. GROUPED PLOTS (by stakeholder / region / etc.)
    # ========================================================
    if not overall_only:
        for grp_col in group_cols:
            print(f"\n\033[1mProcessing {prefix_string} - by {grp_col}\033[0m")

            plot_df = generate_distribution(df_block, grp_col=grp_col)
            if plot_df.empty:
                print(f"No valid responses for {grp_col}.")
                continue

            # In ranked mode, again rename Count → Percent before plotting
            if mode == "ranked":
                plot_df = plot_df.rename(columns={"Count": "Percent"})

            # Merge in overall rows but rename column to grp_col
            plot_df_merged = pd.concat(
                [plot_df_overall.rename(columns={"Overall": grp_col}), plot_df],
                axis=0,
            )

            excel_sheets[grp_col] = plot_df_merged.copy()

            question_ID_group = ", ".join(
                map(str, np.sort(plot_df["Question ID"].dropna().unique()))
            )
            N_number = str(plot_df["N"].max())

            _make_plot(
                plot_df_merged,
                grp_col=grp_col,
                folder_path=folder_path,
                title=(
                    f"Q{question_ID_group}: {prefix_string} "
                    f"(n={N_number}) – by {grp_col}"
                ),
                y_label=grp_col,
                hue_order=hue_order,
                hue_label=hue_label,
                save_png=save_png,
            )

            if save_csv:
                fname = f"Q{OG_question_val}_{grp_col.replace(' ', '_')}_distribution.csv"
                plot_df_merged.to_csv(f"{folder_path}/{fname}", index=False)
                print(f"Saved {grp_col} distribution to: {fname}")

    # ========================================================
    # 7. EXCEL EXPORT (all sheets)
    # ========================================================
    if save_excel:
        excel_path = (
            f"{folder_path}/QG_{safe_folder_name}_all_distributions_Q{OG_question_val}.xlsx"
        )
        with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
            for sheet_name, df_sheet in excel_sheets.items():
                df_sheet.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        print(f"Saved all distributions to: {excel_path}")


In [14]:
# ===========================================================================
#  Select all that apply → absolute count of TRUE per option
# ===========================================================================
def plot_select_all_true_counts(df, prefix_string, save_png=False, outpath=None, show_plot=True):
    """
    Select-all-that-apply:
    Count TRUE responses per option and plot absolute counts.
    """
    df_block = df[df["Question"].astype(str).str.startswith(prefix_string)].copy()
    if df_block.empty:
        print(f"No questions found with prefix: {prefix_string}")
        return None

    df_block = add_option_column(df_block)

    # Normalize answers (your pipeline already does some of this)
    df_block["Answer"] = df_block["Answer"].astype(str).str.strip().str.lower()
    df_block["is_true"] = df_block["Answer"].eq("true")

    counts = (
        df_block.groupby("Option")["is_true"]
        .sum()
        .sort_values(ascending=False)
        .reset_index(name="True Count")
    )

    # ---- Plot ----
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6))
    plt.bar(counts["Option"], counts["True Count"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count of TRUE selections")
    plt.title(f"{prefix_string} — Select-all (TRUE counts)", loc="left", fontweight="bold")
    plt.tight_layout()

    if save_png and outpath:
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        print(f"Saved: {outpath}")

    if show_plot:
        plt.show()
    else:
        plt.close()

    return counts


In [15]:
# =============================================================================
#  Rank in order of preference → convert ranks to points, sum per option, plot
# =============================================================================

def plot_rank_preference_scores(df, prefix_string, points_map=None, save_png=False, outpath=None, show_plot=True):
    """
    Ranking question:
    Convert rank to preference points and aggregate points per option.
    """
    if points_map is None:
        points_map = {"1": 4, "2": 3, "3": 2, "4": 1}

    df_block = df[df["Question"].astype(str).str.startswith(prefix_string)].copy()
    if df_block.empty:
        print(f"No questions found with prefix: {prefix_string}")
        return None

    df_block = add_option_column(df_block)

    # Clean rank strings
    df_block["Answer"] = df_block["Answer"].astype(str).str.strip()
    df_block["Points"] = df_block["Answer"].map(points_map)

    # Keep only valid ranks
    df_block = df_block[df_block["Points"].notna()].copy()

    scores = (
        df_block.groupby("Option")["Points"]
        .sum()
        .sort_values(ascending=False)
        .reset_index(name="Preference Score")
    )

    # ---- Plot ----
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6))
    plt.bar(scores["Option"], scores["Preference Score"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Total preference score (higher = better)")
    plt.title(f"{prefix_string} — Ranking (converted to preference score)", loc="left", fontweight="bold")
    plt.tight_layout()

    if save_png and outpath:
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        print(f"Saved: {outpath}")

    if show_plot:
        plt.show()
    else:
        plt.close()

    return scores


In [16]:
# ================================================================================
#  Matrix Likert (one response per row) → one 100% stacked bar per row (“Option”)
# ================================================================================

def plot_matrix_likert_100pct(df, prefix_string=None, likert_order=None, save_png=False, outpath="../output/quanti_results/", show_plot=True):
    """
    Matrix Likert (select one per row):
    - If prefix_string is provided: filter df to questions that start with that prefix
    - If prefix_string is None: assume df is already the block for the group (as in Step 10)
    Produces one 100% stacked bar per row (Question ID).
    """
    
    if likert_order is None:
        likert_order = ["Strongly disagree", "Somewhat disagree", "Neutral", "Somewhat agree", "Strongly agree"]

    # ------------------------------------------------------------
    # 1) Build df_block safely
    # ------------------------------------------------------------
    if prefix_string is None:
        df_block = df.copy()
    else:
        df_block = df[df["Question"].astype(str).str.startswith(str(prefix_string))].copy()

    if df_block.empty:
        print(f"No rows found for matrix plot. prefix_string={prefix_string}")
        return None

    # ------------------------------------------------------------
    # 2) Ensure Answer is clean (assumes preprocess_answers already ran in Step 10)
    # ------------------------------------------------------------
    df_block["Answer"] = df_block["Answer"].astype(str)

    # ------------------------------------------------------------
    # 3) Build 100% crosstab: one row per Question ID
    # ------------------------------------------------------------
    # counts -> percent within each Question ID
    ctab = pd.crosstab(df_block["Question ID"], df_block["Answer"], normalize="index") * 100

    # keep only likert columns, in correct order (missing ones become 0)
    ctab = ctab.reindex(columns=likert_order, fill_value=0)

    # ------------------------------------------------------------
    # 4) Create labels for each row (Question ID -> prompt after | or full question)
    # ------------------------------------------------------------
    prompts = (
        df_block.groupby("Question ID")["Question"]
        .first()
        .astype(str)
        .str.extract(r"^(.*?)\s*\|")[0]
        .fillna(df_block.groupby("Question ID")["Question"].first().astype(str))
    )
    ctab.index = [f"Q{int(qid)}: {prompts.loc[qid]}" if pd.notna(qid) else str(qid) for qid in ctab.index]

    # ------------------------------------------------------------
    # 5) Plot stacked horizontal bars
    # ------------------------------------------------------------
    ax = ctab.plot(kind="barh", stacked=True, figsize=(12, max(5, 0.35 * len(ctab))))
    ax.set_xlim(0, 100)
    ax.set_xlabel("Responses (%)")
    ax.set_ylabel("")
    ax.set_title("Matrix Likert — 100% stacked distribution", loc="left", fontweight="bold")

    # legend
    ax.legend(loc="lower left", bbox_to_anchor=(0, -0.2), ncol=3, frameon=True)

    plt.tight_layout()

    # ------------------------------------------------------------
    # 6) Save
    # ------------------------------------------------------------
    if save_png and outpath:
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        print(f"Saved: {outpath}")

    if not show_plot:
        plt.close()

    return ctab


## Step 8: Define folder for saving charts

In [17]:
# ============================================================
# STEP 8: Define folder for saving charts
# ============================================================

# Folder where Step 10 & 11 charts and Excel files will be saved
output_folder_charts = "../output/quanti_results/"
os.makedirs(output_folder_charts, exist_ok=True)

## Step 9: Get question groups

In [18]:
# ============================================================
# STEP 9 — Helper: Get Question Groups
# ============================================================

def get_question_groups(df, col="Question Group"):   # Replaced "Mapping Group" with "Question Group"
    """
    Return the unique question groups from the given column.

    Parameters
    ----------
    df : pd.DataFrame
        Main quantitative dataframe.
    col : str
        Column name containing question group identifiers.

    Returns
    -------
    np.ndarray
        Array of unique, non-null question groups.
    """
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found in DataFrame.")

    groups = df[col].dropna().unique()
    print(f"📘 Found {len(groups)} question groups in '{col}':")
    display(groups)
    return groups


# ------------------------------------------------------------
# Execute: get question groups once and reuse
# ------------------------------------------------------------
question_groups = get_question_groups(df, col="Question Group")

# Keep a separate copy if you still want test_array like original code
test_array = question_groups.copy()


📘 Found 51 question groups in 'Question Group':


array(['17,18', '19,20', '21,22', '23,24', '25,26', '27,28', 29, 31, 32,
       '33,34', 35, 36, 37, 38, 39, 40, 41, 42, 43, '44,45', '46,47', 48,
       '49,50', '51,52', '53,54', 55, 56, 57, 58, 60, 61, '63,64,65,66',
       67, 68, 69, 70, 71, 72, 73, '74,75', 76, 77, 78, 79, 80, 82,
       '84,85', '86,87', '88,89', '90,91', 92], dtype=object)

In [19]:
# ============================================================
# Utility Demo — How pd.concat behaves (rows vs columns)
# ============================================================

def demo_concat_behavior():
    """Small demo illustrating row-wise vs column-wise concatenation."""
    import pandas as pd

    df1 = pd.DataFrame({'A': ['A0', 'A1'], 'B': ['B0', 'B1']})
    df2 = pd.DataFrame({'A': ['A2', 'A3'], 'B': ['B2', 'B3']})
    df3 = pd.DataFrame({'C': ['C0', 'C1'], 'D': ['D0', 'D1']})

    # Row-wise concatenation (default axis=0)
    result_rows = pd.concat([df1, df2])
    print("Row-wise concatenation:\n", result_rows)

    # Column-wise concatenation (axis=1)
    result_cols = pd.concat([df1, df3], axis=1)
    print("\nColumn-wise concatenation:\n", result_cols)

    # Row-wise concatenation with new continuous index
    result_ignore_index = pd.concat([df1, df2], ignore_index=True)
    print("\nRow-wise concatenation (ignore_index=True):\n", result_ignore_index)


# Call only if you want to see the demo
# demo_concat_behavior()


## Step 10: Plot question groups with Group Quanti? == 'n' questions

In [20]:
# ============================================================
# STEP 10 — Plot Question Groups with No Grouped Quantitative Questions
# ============================================================

def analyse_non_group_quanti_question_groups(df, output_folder_charts="../output/quanti_results/"):
    """
    Analyse and plot all question groups where `Group Quanti?` == 'n'.

    Adds support for:
      1) Select-all-that-apply  -> plot_select_all_true_counts
      2) Ranking (1..4)         -> plot_rank_preference_scores (rank->points mapping)
      3) Matrix Likert          -> plot_matrix_likert_100pct (one bar per row)
    """
    global open_ended_answers  # used inside preprocess_answers

    os.makedirs(output_folder_charts, exist_ok=True)

    # --------------------------------------------------------
    # 1) Filter to question groups with Group Quanti? == 'n'
    # --------------------------------------------------------
    group_n = df[df["Group Quanti?"] == "n"].copy()

    if group_n.empty:
        print("⚠️ No rows found where Group Quanti? == 'n'")
        return pd.DataFrame()

    question_groups = group_n["Question Group"].dropna().unique()
    print(f"🔍 Found {len(question_groups)} question groups with Group Quanti? == 'n'")
    display(question_groups)

    df_claims_group = pd.DataFrame()

    # --------------------------------------------------------
    # Helper: detect question type from Answer patterns
    # --------------------------------------------------------
    def _detect_question_type(df_q):
        vals = (
            df_q["Answer"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )

        # Select-all (boolean)
        if set(vals).issubset({"True", "False"}):
            return "select_all"

        # Rank (1..4) (or 1..5 if your ranks vary)
        if all(v.isdigit() for v in vals) and set(vals).issubset({"1", "2", "3", "4"}):
            return "rank"

        # Otherwise, treat as likert / categorical
        return "likert_or_other"

    # --------------------------------------------------------
    # 2) Loop over each question group
    # --------------------------------------------------------
    for group in question_groups:
        print(f"\n\033[1mAnalyzing Question Group: {group}\033[0m")

        df_group = group_n[group_n["Question Group"] == group].copy()
        if df_group.empty:
            continue

        # All Question IDs in this group (sorted)
        question_ids = (
            df_group["Question ID"]
            .dropna()
            .unique()
            .astype(int)
        )
        question_ids = sorted(question_ids)

        # For matrix-likert style groups, we’ll accumulate the per-row overall distributions
        df_overall_group = pd.DataFrame()
        df_overall_abs_group = pd.DataFrame()

        # ----------------------------------------------------
        # 2a) Loop over each Question ID inside the group
        # ----------------------------------------------------
        per_qtype = []  # track detected type per qid (for group-level decision)

        for qid in question_ids:
            df_q = df_group[df_group["Question ID"] == qid].reset_index(drop=True)

            # Debug/inspection
            print_question_options(df_q)

            # Normalize answers (your enhanced preprocess)
            df_q = preprocess_answers(df_q, qid=qid)

            # Detect question type AFTER preprocessing
            qtype_detected = _detect_question_type(df_q)
            per_qtype.append(qtype_detected)

            # Quick distribution table
            show_answer_distribution(df_q, save_csv=False)

            # ---- Route to your new plots where applicable ----
            # (These plots are usually clearer than forcing everything into show_metadata_breakdown)
            if qtype_detected == "select_all":
                # Select-all: count TRUE per option (absolute counts)
                # Typically df_q includes Option-level rows via the Question text after "|"
                plot_select_all_true_counts(
                    df_q=df_q,
                    output_folder=output_folder_charts,
                    save_png=True,
                    show_plot=True
                )
                continue

            if qtype_detected == "rank":
                # Ranking: convert ranks -> preference points and sum
                plot_rank_preference_scores(
                    df_q=df_q,
                    output_folder=output_folder_charts,
                    save_png=True,
                    show_plot=True
                )
                continue

            # Otherwise keep original logic for likert/categorical:
            (
                df_overall,
                df_overall_abs,
                is_likert,
                order,
                colors,
                folder_path,
            ) = show_metadata_breakdown(
                df_q,
                plot_overall_only=False,
                show_labels=True,
                save_csv=False,
                save_png=True,
                save_excel=True,
                show_plot=True,                 # <-- changed so you can SEE plots
                only_stake_category=False,
                output_folder=output_folder_charts,
            )

            df_overall_group = pd.concat([df_overall, df_overall_group], axis=0, ignore_index=True)
            df_overall_abs_group = pd.concat([df_overall_abs, df_overall_abs_group], axis=0, ignore_index=True)

        # ----------------------------------------------------
        # 2b) Group-level plotting (Matrix Likert groups)
        # ----------------------------------------------------
        # If the group looks like multiple likert sub-rows, plot a single 100% stacked chart (one bar per row).
        # Rule of thumb: more than 1 qid AND most are likert_or_other.
        if len(question_ids) > 1 and per_qtype.count("likert_or_other") >= 2:
            plot_matrix_likert_100pct(
                df_group.copy(),
                prefix_string=None,
                likert_order=None,
                save_png=True,
                outpath=output_folder_charts,
                show_plot=True
            )

        # ----------------------------------------------------
        # 2c) (Optional) Keep your original “summary across sub-questions”
        #     ONLY when df_overall_group exists (i.e., we ran show_metadata_breakdown)
        # ----------------------------------------------------
        if len(df_overall_group) > 1:
            if group != "226,227,228,229,230,231":
                group_trimmed = ",".join(str(group).split(",")[:-1])
            else:
                group_trimmed = ",".join(str(group).split(","))

            question_id_str = group_trimmed
            min_index = df_group.index.min()
            question_id_og_str = df_group.loc[min_index, "OG Question Number"]

            df_group["Prompt"] = (
                df_group["Question"]
                .str.extract(r"^(.*?)\s*\|")[0]
                .fillna(df_group["Question"])
            )
            question_str = df_group.loc[min_index, "Prompt"]

            plot_likert_bars(
                df_overall_group,
                col="overall",
                title=f"Q{question_id_og_str}: {question_str}",
                colors=colors,
                folder_path=folder_path,
                question_id_str=question_id_str,
                question_id_og_str=question_id_og_str,
                is_likert=is_likert,
                is_meta=True,
                show_labels=True,
                labels=[shorten_label_2(l) for l in order],
                save_png=True,
                fname=f"Q{question_id_og_str}_group_overall_distribution_Q{question_id_str}",
            )

    # --------------------------------------------------------
    # 3) Export open-ended answers
    # --------------------------------------------------------
    display(open_ended_answers)
    os.makedirs("../output", exist_ok=True)
    open_ended_answers.to_csv("../output/open_ended_answers_by_question.csv", index=False)
    print("💾 Saved open-ended answers to: ../output/open_ended_answers_by_question.csv")

    return df_claims_group



In [21]:
# ------------------------------------------------------------
# RUN STEP 10
# ------------------------------------------------------------
output_folder_charts = "../output/quanti_results/"
df_claims_group = analyse_non_group_quanti_question_groups(df=df, output_folder_charts=output_folder_charts)


🔍 Found 29 question groups with Group Quanti? == 'n'


array(['17,18', '19,20', '21,22', '23,24', '25,26', '27,28', 29, '33,34',
       35, 36, 39, 42, '44,45', '46,47', 48, '49,50', '51,52', '53,54',
       61, '63,64,65,66', 67, 68, 69, 72, 73, 76, 79, 80, '84,85'],
      dtype=object)


Analyzing Question Group: 17,18


NameError: name 'parse_question_options' is not defined

In [22]:
print("Answer distribution after preprocess:")
print(df_q["Answer"].value_counts(dropna=False).head(10))


Answer distribution after preprocess:


NameError: name 'df_q' is not defined

# Step 11: Plot question groups with grouped quantitative questions